In [1]:
# 1단계: 학습된 카테고리 모델 로드
import joblib
import pandas as pd

model = joblib.load('models/category_cf_model_pc12.pkl')
if model.get('model_type') != 'category_cf_v1':
    raise ValueError('잘못된 모델 파일입니다. knn01_카테고리기반.ipynb를 다시 실행하세요.')

category_similarity_df = model['category_similarity_df']
popular_categories = model['popular_categories']
print('불러온 카테고리:', category_similarity_df.index.tolist())


불러온 카테고리: ['CASE', 'CPU', 'GPU', 'HDD', 'KEYBOARD', 'MAINBOARD', 'MONITOR', 'MOUSE', 'POWER', 'RAM', 'SPEAKER', 'SSD']


In [2]:
# 2단계: DB 조회로 받은 구매 카테고리를 입력받는 추천 함수
def recommend(purchased_categories, top_n=5):
    categories = [str(x).strip().upper() for x in purchased_categories]
    unknown = sorted(set(categories) - set(category_similarity_df.index))
    if unknown:
        raise ValueError(f'모델에 없는 카테고리입니다: {unknown}')

    if not categories:
        scores = popular_categories.astype(float)
        scores = scores / scores.max()
        recommend_type = 'POPULAR'
    else:
        scores = pd.Series(0.0, index=category_similarity_df.index)
        for category in set(categories):
            scores += category_similarity_df[category]
        scores = scores[scores > 0].sort_values(ascending=False)
        if scores.empty:
            scores = popular_categories.astype(float)
            scores = scores / scores.max()
            recommend_type = 'POPULAR'
        else:
            scores = scores / scores.max()
            recommend_type = 'PERSONALIZED'

    top = scores.sort_values(ascending=False).head(top_n)
    return pd.DataFrame({
        'category': top.index.to_list(),
        'recommend_score': top.values,
        'recommend_type': recommend_type
    })


In [3]:
# 3단계: 기존 회원과 신규 회원 테스트
# 나중에 Repository가 실제 회원의 주문을 조사해 ['CPU', 'RAM'] 같은 목록을 전달합니다.
print('기존 회원 (CPU와 RAM 구매):')
print(recommend(['CPU', 'RAM'], top_n=5).to_string(index=False))

print('신규 회원 (구매 이력 없음):')
print(recommend([], top_n=5).to_string(index=False))

# CSV 가상 상품 ID를 어떤 숫자로 바꾸든, 이 모델의 입력에는 상품 ID가 없습니다.
assert 'product_df' not in model
assert 'user_item_matrix' not in model
print('검증 통과: 모델이 가상 상품·회원 ID에 의존하지 않습니다.')


기존 회원 (CPU와 RAM 구매):
 category  recommend_score recommend_type
MAINBOARD         1.000000   PERSONALIZED
      GPU         0.983901   PERSONALIZED
  MONITOR         0.902442   PERSONALIZED
      SSD         0.890260   PERSONALIZED
 KEYBOARD         0.856590   PERSONALIZED
신규 회원 (구매 이력 없음):
category  recommend_score recommend_type
 MONITOR         1.000000        POPULAR
     CPU         0.882518        POPULAR
KEYBOARD         0.869028        POPULAR
     GPU         0.774592        POPULAR
     SSD         0.750422        POPULAR
검증 통과: 모델이 가상 상품·회원 ID에 의존하지 않습니다.
